# 06 · Pipeline de Preprocesamiento para Producción
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Transformar el dataset crudo depurado en una **matriz numérica definitiva lista para modelar**, encapsulando todas las reglas de negocio en un objeto de Scikit-Learn (`ColumnTransformer`). 

Este objeto se entrenará (`fit`) **estrictamente sobre TRAIN** y luego se proyectará (`transform`) sobre TRAIN y TEST.

### 🧠 Arquitectura del Pipeline:
1. **Selección Dinámica:** Consumiremos el reporte del Cuaderno 05 (`iv_table.json`) para descartar automáticamente las variables inútiles.
2. **Tratamiento Diferenciado de Nulos:**
   * **Nulos MNAR (Informativos):** Las variables `age` y `falsification_of_information` se imputarán con la constante `"FALTANTE_MNAR"`, preservando el riesgo que descubrimos en el EDA.
   * **Nulos Aleatorios (Resto):** Se imputarán utilizando la Moda (`most_frequent`) aprendida del conjunto de entrenamiento.
3. **Codificación Robusta:** Aplicaremos *One-Hot Encoding* con el parámetro `handle_unknown='ignore'`. Si en el futuro un usuario ingresa una categoría nueva que el modelo nunca vio, el pipeline la ignorará elegantemente en lugar de colapsar.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA DE DATOS (MÓDULO 04) Y SELECCIÓN POR IV (MÓDULO 05)
# ==============================================================================
import pandas as pd
import numpy as np
import json
import joblib

# Configuración visual
pd.set_option('display.max_columns', None)

# 1. Cargar datasets depurados de outliers (Módulo 04)
train_raw = pd.read_csv('data_processed/04_train_ecommerce.csv')
test_raw = pd.read_csv('data_processed/04_test_ecommerce.csv')

target_col = 'Cluster'

# 2. Cargar el reporte de Information Value (Módulo 05) para purgar ruido
with open('reports/iv_table.json', 'r') as f:
    reporte_iv = json.load(f)

# Extraemos la lista de variables inútiles (IV < 0.02)
cols_a_descartar = reporte_iv['variables_descartadas_iv_bajo']

print(f"🗑️ Descartando {len(cols_a_descartar)} variables inútiles según el reporte de IV...")

# 3. Separar X e y filtrando las columnas malas
X_train = train_raw.drop(columns=[target_col] + cols_a_descartar, errors='ignore')
y_train = train_raw[target_col]

X_test = test_raw.drop(columns=[target_col] + cols_a_descartar, errors='ignore')
y_test = test_raw[target_col]

print(f"📦 X_train listo para el Pipeline: {X_train.shape[0]} filas × {X_train.shape[1]} variables útiles.")

🗑️ Descartando 15 variables inútiles según el reporte de IV...
📦 X_train listo para el Pipeline: 8018 filas × 46 variables útiles.


## 2. Construcción del `ColumnTransformer` (El Motor del Pipeline)

Separaremos nuestras variables en dos ramas (*Transformers*):
* **Rama MNAR:** Imputación Constante + One-Hot Encoding.
* **Rama Normal:** Imputación por Moda + One-Hot Encoding.

El `ColumnTransformer` unirá ambas ramas en una sola matriz de forma paralela y ultra-optimizada.

In [2]:
# ==============================================================================
# BLOQUE 2: CONSTRUCCIÓN DEL PIPELINE DE PREPROCESAMIENTO (SCIKIT-LEARN)
# ==============================================================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 1. Definición de grupos de variables según diagnóstico del EDA
vars_mnar = ['age', 'falsification_of_information']
# Validamos que estén en X_train (por si alguna hubiera sido descartada)
vars_mnar = [col for col in vars_mnar if col in X_train.columns]

vars_normales = [col for col in X_train.columns if col not in vars_mnar]

print(f"🔧 Configurando motor: {len(vars_mnar)} variables MNAR | {len(vars_normales)} variables normales.")

# 2. Construcción de Ramas (Pipelines individuales)
rama_mnar = Pipeline(steps=[
    ('imputer_constante', SimpleImputer(strategy='constant', fill_value='FALTANTE_MNAR')),
    ('ohe_mnar', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

rama_normal = Pipeline(steps=[
    ('imputer_moda', SimpleImputer(strategy='most_frequent')),
    ('ohe_normal', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Ensamblaje del Preprocesador Maestro
preprocesador = ColumnTransformer(transformers=[
    ('mnar_pipe', rama_mnar, vars_mnar),
    ('normal_pipe', rama_normal, vars_normales)
], remainder='drop')  # 'drop' por seguridad, aunque ya pasamos todas las columnas

# ------------------------------------------------------------------------------
# 4. LA REGLA DE ORO: FIT SOBRE TRAIN, TRANSFORM SOBRE AMBOS
# ------------------------------------------------------------------------------
print("⚙️ Entrenando Pipeline sobre TRAIN y transformando matrices...")

# Ajustamos (fit) y transformamos X_train
X_train_proc_array = preprocesador.fit_transform(X_train)

# SOLO transformamos X_test (sin fit)
X_test_proc_array = preprocesador.transform(X_test)

# Extraemos los nombres de las nuevas columnas generadas por el One-Hot Encoder
nombres_features = preprocesador.get_feature_names_out()

# Reconstruimos los DataFrames para facilitar su uso posterior
X_train_proc = pd.DataFrame(X_train_proc_array, columns=nombres_features, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc_array, columns=nombres_features, index=X_test.index)

print(f"✅ Transformación exitosa.")
print(f"· Dimensiones finales Train: {X_train_proc.shape[0]} filas × {X_train_proc.shape[1]} features.")
print(f"· Dimensiones finales Test:  {X_test_proc.shape[0]} filas × {X_test_proc.shape[1]} features.")

🔧 Configurando motor: 2 variables MNAR | 44 variables normales.
⚙️ Entrenando Pipeline sobre TRAIN y transformando matrices...
✅ Transformación exitosa.
· Dimensiones finales Train: 8018 filas × 212 features.
· Dimensiones finales Test:  2022 filas × 212 features.


## 3. Tests de Integridad y Exportación a Producción

En ingeniería de Machine Learning, no asumimos que los datos están limpios; **lo demostramos con código**. Utilizaremos funciones `assert` para garantizar que la transformación no dejó fisuras.

Si los tests pasan, extraeremos el objeto `preprocesador` como un archivo `.joblib`. Este archivo contiene las "reglas matemáticas" (las modas aprendidas, las columnas dummy generadas) y es el artefacto que los ingenieros de software integrarían en un servidor para predecir en tiempo real.

In [3]:
# ==============================================================================
# BLOQUE 3: TESTS DE INTEGRIDAD Y EXPORTACIÓN DE ARTEFACTOS
# ==============================================================================
import joblib
import os

# 1. Tests de Integridad (Asserts) - Si algo falla, el código se detendrá con error
print("🧪 Ejecutando tests de integridad...")
assert list(X_train_proc.columns) == list(X_test_proc.columns), "Error: Las columnas de Train y Test no coinciden."
assert X_train_proc.isna().sum().sum() == 0, "Error: Quedan valores nulos en Train tras el pipeline."
assert X_test_proc.isna().sum().sum() == 0, "Error: Quedan valores nulos en Test tras el pipeline."
print("✅ Todos los tests pasados con éxito. Matrices 100% íntegras y sin nulos.")

# 2. Reintegrar el Target para guardar las bases finales listas para modelar
train_final = X_train_proc.copy()
train_final[target_col] = y_train.values

test_final = X_test_proc.copy()
test_final[target_col] = y_test.values

# 3. Exportación a disco
os.makedirs('data_processed', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

# Guardamos las bases procesadas
train_final.to_csv('data_processed/06_train_proc.csv', index=False)
test_final.to_csv('data_processed/06_test_proc.csv', index=False)

# Guardamos el objeto Pipeline (El Motor de Producción)
joblib.dump(preprocesador, 'artifacts/preprocesador.joblib')

print("\n" + "="*75)
print("🚀 PIPELINE DE PRODUCCIÓN Y MATRICES EXPORTADOS CON ÉXITO:")
print(f"· Base final Train: 'data_processed/06_train_proc.csv'")
print(f"· Base final Test:  'data_processed/06_test_proc.csv'")
print(f"· Objeto Pipeline:  'artifacts/preprocesador.joblib' (Listo para el despliegue)")
print("="*75)

🧪 Ejecutando tests de integridad...
✅ Todos los tests pasados con éxito. Matrices 100% íntegras y sin nulos.

🚀 PIPELINE DE PRODUCCIÓN Y MATRICES EXPORTADOS CON ÉXITO:
· Base final Train: 'data_processed/06_train_proc.csv'
· Base final Test:  'data_processed/06_test_proc.csv'
· Objeto Pipeline:  'artifacts/preprocesador.joblib' (Listo para el despliegue)


## 4. Conclusiones del Pipeline de Preprocesamiento

La construcción del `ColumnTransformer` marca la transición exitosa de una base de datos cualitativa hacia un entorno matricial de Machine Learning puro. Las decisiones arquitectónicas adoptadas garantizan la robustez del modelo predictivo:

---

### A. Blindaje contra Data Leakage (Fuga de Información)
* **Aislamiento Metodológico:** El pipeline fue ajustado (`fit`) única y exclusivamente sobre los datos de entrenamiento ($Train$). Parámetros como la categoría más frecuente (Moda) o las estructuras del *One-Hot Encoding* se aprendieron sin observar jamás la muestra de validación ($Test$).
* **Proyección Segura:** Al utilizar el método `.transform()` sobre $Test$, garantizamos que el modelo evalúe observaciones nuevas bajo las mismas reglas matemáticas, simulando con fidelidad un entorno de producción en tiempo real.

### B. Tratamiento Estratégico de Ausencias
* **Conservación de Señal MNAR:** Las variables con omisión informativa (como `age` y `falsification_of_information`) no fueron diluidas con imputaciones por moda. El uso de la constante `"FALTANTE_MNAR"` preserva el comportamiento evasivo del usuario como un predictor fuerte de su aversión al riesgo.
* **Resiliencia ante lo Desconocido:** La configuración `handle_unknown='ignore'` protege al servidor de caídas en producción frente a la eventual aparición de perfiles demográficos inéditos.

---

### 🏆 Cierre de la Fase de Preprocesamiento
Con el objeto `preprocesador.joblib` guardado y las matrices finales de $212$ dimensiones ($0$ nulos, validación de integridad superada), la fase de preprocesamiento de datos concluye oficialmente. 

El proyecto cuenta ahora con la matriz expandida necesaria para proceder al cuaderno **`07_seleccion_vars.ipynb`**, donde aplicaremos un **Consenso Multicriterio de Selección de Variables** (Información Mutua, VIF y Random Forest) para reducir estas $212$ características a un subconjunto óptimo y compacto ($\sim 30 \text{ - } 35$ features) antes del modelado supervisado.